# Person C — Train the LSTM in Google Colab

Train and compare context lengths 3, 5, and 10. Select **Runtime → Change runtime type → T4 GPU** before running the full experiments. Outputs are saved to Google Drive.

> **Before running:** download `colab/person_c_training_code.zip` from this project. The notebook will ask you to upload it if the GitHub clone does not contain the local Person C files yet.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/junaidjmomin/DeepLearning-RNNvsLSTMvsANN.git'
REPO_DIR = Path('/content/DeepLearning-RNNvsLSTMvsANN')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if not Path('src/training/train_lstm.py').is_file():
    from google.colab import files
    from zipfile import ZipFile
    zip_candidates = list(Path('/content').rglob('person_c_training_code.zip'))
    if not zip_candidates:
        uploaded = files.upload()  # Or use VS Code Explorer: Upload to Colab
        zip_candidates = [Path(name) for name in uploaded if name.endswith('.zip')]
    assert zip_candidates, 'Upload the person_c_training_code.zip bundle.'
    with ZipFile(zip_candidates[0]) as archive:
        archive.extractall(REPO_DIR)
assert Path('src/training/train_lstm.py').is_file(), 'The training files were not installed in the project folder.'
for name in ['train.json', 'validation.json', 'test.json', 'vocab.json']:
    assert (Path('data/processed') / name).is_file(), f'Missing processed file: {name}'
print('Project and processed DailyDialog data are ready:', Path.cwd())

In [ ]:
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow>=2.15,<2.20', 'numpy>=1.24,<3'], check=True)
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('No GPU detected; full training will take longer. Enable a GPU runtime in Runtime settings.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/ContextNext_PersonC')
ARTIFACT_DIR = OUTPUT_DIR / 'artifacts'
RESULTS_DIR = OUTPUT_DIR / 'results'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive output folder:', OUTPUT_DIR)

## Training configuration

These settings match the current RNN baseline. Keep them fixed across context lengths. Run each context cell separately; results are written to Drive after each run.

In [ ]:
def train_lstm_context(context_words, epochs=20):
    cmd = [sys.executable, '-m', 'src.training.train_lstm',
           '--sequence-length', str(context_words), '--embedding-dim', '128',
           '--hidden-units', '256', '--dropout', '0.2', '--recurrent-dropout', '0.0',
           '--learning-rate', '0.001', '--batch-size', '256', '--epochs', str(epochs),
           '--patience', '3', '--seed', '42', '--artifact-dir', str(ARTIFACT_DIR),
           '--results-dir', str(RESULTS_DIR)]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

# Optional quick smoke run instead of a full experiment:
# subprocess.run([sys.executable, '-m', 'src.training.train_lstm', '--sequence-length', '3', '--epochs', '1', '--max-train-samples', '2000', '--max-validation-samples', '500', '--max-test-samples', '500', '--artifact-dir', str(ARTIFACT_DIR), '--results-dir', str(RESULTS_DIR)], cwd=REPO_DIR, check=True)

### Context length: 3

In [ ]:
train_lstm_context(3)

### Context length: 5

In [ ]:
train_lstm_context(5)

### Context length: 10

In [ ]:
train_lstm_context(10)

## Compare the saved runs

In [ ]:
import json, pandas as pd
files = sorted(RESULTS_DIR.glob('lstm*_test_metrics.json'))
rows = [json.loads(path.read_text()) for path in files]
keys = ['sequence_length', 'top_1_accuracy', 'top_3_accuracy', 'cross_entropy', 'perplexity', 'mse', 'mae', 'rmse', 'parameter_count', 'training_seconds', 'inference_latency_ms']
if rows:
    display(pd.DataFrame([{key: row.get(key) for key in keys} for row in rows]).sort_values('sequence_length'))
else:
    print('No result files yet. Run the three context training cells above.')

## Export examples for error analysis

In [ ]:
cmd = [sys.executable, '-m', 'src.training.analyze_predictions',
       '--model', str(ARTIFACT_DIR / 'models' / 'lstm_best.keras'),
       '--sequence-length', '3', '--output', str(RESULTS_DIR / 'lstm_prediction_examples.json')]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Prediction examples saved to:', RESULTS_DIR / 'lstm_prediction_examples.json')

## Bring the outputs back to the project

The checkpoints and metrics are saved in Google Drive at `MyDrive/ContextNext_PersonC/`. Copy the files under `artifacts/models/` and `results/` back into those folders in the local project. The app's default LSTM prediction uses the 3-word checkpoint `lstm_best.keras`; the 5- and 10-word runs have separate checkpoint names.